# Loan portfolio & rating migration framework (synthetic)

| File | Content |
|---|---|
| `matrices.py` | `MigrationMatrices`: loads the matrix table (SQL format), gives annual and quarterly matrices |
| `migration.py` | Deterministic quarterly migration of the portfolio amount vectors, with amortisation |
| `visualization.py` | Styled pandas views and interactive explorers |
| `excel_export.py` | Excel report: matrices and the quarter-by-quarter migration of the portfolio |
| `rating_models.py` | *Synthetic only*: generates the matrices and returns them in the SQL table format |
| `portfolio.py` | *Synthetic only*: generates the loan portfolio |

**Flow**: matrix table (`model, scenario, year, rank, 1..14`) → `matrices = MigrationMatrices.from_frame(table)`
→ `project_portfolio(portfolio, matrices, scenario)` → `export_workbook(...)`.

**At work**, only two cells change: the matrix table (SQL query instead of the synthetic generator)
and the portfolio (your loans instead of `generate_portfolio`). Everything after them is unchanged.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from matrices import MigrationMatrices
from migration import REQUIRED_COLUMNS, initial_vectors, project_portfolio, scenario_comparison, validate_portfolio
from visualization import (
    matrix_explorer, migration_explorer, portfolio_explorer, portfolio_summary, style_count_grid,
    style_exposure_grid, style_flows, style_initial_vectors, style_matrix, style_matrix_diff, style_matrix_table,
    style_model_summary, style_pd_table, style_portfolio, style_quarterly_matrix, style_scenario_comparison,
    style_vectors,
)
# synthetic data only
from rating_models import MODEL_SPECS, SEED, generate_models, to_matrix_table
from portfolio import REFERENCE_DATE, generate_portfolio, loan_matrix, model_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

rng = np.random.default_rng(SEED)  # one generator for matrices then portfolio -> reproducible

## 1. Migration matrices

### Matrix table (= SQL extract)

One row per **model / scenario / year of application / starting rank**, then the 14 class probabilities.
A 10-class model has 0 (or NULL) in columns 11-14; its last class holding a probability is its Default class.

*Synthetic*: TTC matrices built parametrically, then stressed per scenario / year with a one-factor Vasicek Z-shift
(parameters in `MODEL_SPECS` / `Z_PATHS`, `rating_models.py`).

In [ ]:
# >>> At work: matrix_table = pd.read_sql(QUERY, connection)
models = generate_models(MODEL_SPECS, rng)
matrix_table = to_matrix_table(models)

style_matrix_table(matrix_table, rows=15)

### Matrices object

`MigrationMatrices.from_frame` checks the table (missing / duplicate ranks, rows summing to 1, % or decimals)
and builds one matrix per model / scenario / year. If your column names differ, map them:
`MigrationMatrices.from_frame(matrix_table, model="MODEL_NAME", scenario="SCENARIO", year="YEAR", rank="RANK")`.

| | |
|---|---|
| `matrices.annual(model, scenario, year)` | annual matrix (numpy); years after the last one use the last one (Y3+) |
| `matrices.quarterly(model, scenario, quarter)` | quarterly matrix for projection quarter `quarter` (4th root of that year's matrix) |
| `matrices.annual_df(...)` / `quarterly_df(...)` | same as labelled DataFrames |
| `matrices.n_buckets[model]`, `labels(model)`, `models`, `scenarios()` | structure |

In [ ]:
matrices = MigrationMatrices.from_frame(matrix_table)
matrices

### 1-year PD by class, scenario and year

In [ ]:
for m in matrices.models:
    display(style_pd_table(matrices, m))

### Annual matrices

Rows = from, columns = to. Blue = stay, green = upgrade, orange = downgrade, red = default.

In [ ]:
display(style_matrix(matrices, "CORP_14", "adverse", 1))
display(style_matrix(matrices, "SME_10", "severely_adverse", 3))

### Stress vs base (pp)

In [ ]:
style_matrix_diff(matrices, "SME_10", "severely_adverse", 1)

### Quarterly matrices

`Q` is the 4th root of the annual matrix (`Q^4 = M`): quarters 1-4 use year 1, 5-8 year 2, 9+ year 3+.
(Should a root have negative entries, they are set to 0 and taken off the diagonal.)

In [ ]:
display(matrices.root_errors().style.format("{:.1e}").set_caption("max |Q^4 - annual matrix| (pp)"))
style_quarterly_matrix(matrices, "CORP_14", "adverse", 1)

### Matrix explorer

In [ ]:
matrix_explorer(matrices)

## 2. Loan portfolio

Columns needed for the migration: `rating_model` (same names as in the matrix table), `rating` (performing class),
`amount`, `maturity_years` (residual maturity in years). `validate_portfolio` checks them against the matrices.

In [ ]:
# >>> At work: portfolio = <your loans with the REQUIRED_COLUMNS>
portfolio = generate_portfolio(models, rng=rng)
validate_portfolio(portfolio, matrices)

style_portfolio(portfolio.head(15))

### Summary by model and rating

In [ ]:
display(style_model_summary(model_summary(portfolio)))
display(portfolio_summary(portfolio))

### Rating x residual maturity

In [ ]:
for m in matrices.models:
    display(style_exposure_grid(portfolio, matrices, m))
    display(style_count_grid(portfolio, matrices, m))

### Matrix attached to a loan

In [ ]:
# First loan, adverse scenario, projection year 5 (-> year 3+ matrix)
lid = portfolio.index[0]
row = portfolio.at[lid, "rating"] - 1
print(lid, portfolio.loc[lid, ["rating_model", "rating"]].to_dict())
print("Transition row (adverse, Y3+, %):", (loan_matrix(portfolio, matrices, lid, "adverse", 5)[row] * 100).round(2))

### Portfolio explorer

In [ ]:
portfolio_explorer(portfolio, matrices)

## 3. Quarterly migration

**Method** (`migration.py`)
* The portfolio is aggregated into **one amount vector per rating model** (classes 1..n-1 + `D`).
* Each quarter: **1) migrate** the vector with the quarterly matrix, **2) amortise** the performing classes.
  Default is absorbing and not amortised.
* **Amortisation** follows the loans' maturity dates: each vector is split into maturity cohorts (loans maturing in the
  same quarter) migrated separately, so repayments remove exactly what is left of those loans, whatever class it has moved to.
  * `bullet`: full repayment at the end of the maturity quarter.
  * `linear`: straight-line repayment down to 0 at the maturity quarter.

### Starting vectors

The portfolio aggregated by rating model and class: each vector's total equals the amount rated by that model.

In [ ]:
vectors_0 = initial_vectors(portfolio, matrices)
for name, v in vectors_0.items():
    assert np.isclose(v.sum(), portfolio.loc[portfolio.rating_model == name, "amount"].sum())
style_initial_vectors(vectors_0)

### Vectors migrating quarter by quarter

Each column is the model's vector at the end of that quarter (Q0 = today). Amounts leave the performing classes
through defaults (into `D`) and repayments (`Repaid (cum.)`). `Total` stays equal to the initial amount.

In [ ]:
SCENARIO = "adverse"
AMORTIZATION = "bullet"   # or "linear"

projections = project_portfolio(portfolio, matrices, SCENARIO, AMORTIZATION, reference_date=REFERENCE_DATE)
for p in projections.values():
    display(style_vectors(p))

### Quarterly flows

Per quarter: opening performing amount, new defaults, repayments, closing performing amount,
cumulative defaults / repayments and default rates. `check` must be 0.

In [ ]:
for p in projections.values():
    display(style_flows(p))

### Scenario comparison

Cumulative defaults as a share of each model's initial amount.

In [ ]:
style_scenario_comparison(scenario_comparison(portfolio, matrices, AMORTIZATION),
                          f"Cumulative default rate (% of initial amount) - {AMORTIZATION}")

### Migration explorer

Any model / scenario / amortisation type, as vectors or flows.

In [ ]:
migration_explorer(portfolio, matrices, REFERENCE_DATE)

## 4. Excel report

For the scenario used (`SCENARIO`), per rating model:

| Sheet | Content |
|---|---|
| `<model> matrices` | Annual matrix of years 1, 2, 3+ and the quarterly matrix applied in the projection |
| `<model> portfolio` | Amount per class at the end of each quarter, cumulative repayments, total |

In [ ]:
from pathlib import Path
from excel_export import export_workbook

out = Path("output") / "migration_report.xlsx"
out.parent.mkdir(exist_ok=True)
export_workbook(out, matrices, portfolio, SCENARIO, AMORTIZATION, reference_date=REFERENCE_DATE)
print(f"Written: {out.resolve()}")

## Next steps
* Apply LGD to the default flows to get losses per quarter and scenario.